### Realiza a segmentação do que deve ser Faturado ou não ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador" 

In [0]:
dbutils.widgets.text("Mes", "", "Data Bases Ativo e Encerrrado")
Mes = dbutils.widgets.get("Mes")

In [0]:
base_ger = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Final_Mensal/CIVEL_GERENCIAL - {Mes}.xlsx'
base_ac = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Aceite_Mensal/Data Aceite - Consolidada.xlsx'
base_fatu = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Faturamento_honoararios_2023 - 2025/Consolidado Faturamento Honorários Cível Massa 2023 e 2025.xlsx'
base_zurick = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Zurick/PJ - REGRAS DO FLUXO DE CADASTRO CÍVEL MASSA - {Mes}.xlsx'

In [0]:
df_ger = pd.read_excel(
    base_ger,
    sheet_name='CÍVEL',
    header=5
)

df_ac = pd.read_excel(
    base_ac,
    sheet_name='ACEITE DE PATROCÍNIO E DESCREDE',
    header=5
)

#df_fatu_hist = pd.read_excel(
 #   base_fatu,
  #  sheet_name='Contrato Novo',
   # header=1
#)

df_fatu_hist_old = pd.read_excel(
    base_fatu,
    sheet_name='Contrato Antigo',
    header=1
)


In [0]:
df_id_zurick = pd.read_excel(
    base_zurick,
    sheet_name='REGRAS DO FLUXO DE CAD CIVEL',
    header=5
)

In [0]:
# 1. Defina as condições
condicao_1 = df_ger['TERCEIRO SOLVENTE'] == 'ZURICH'
#condicao_2 = df_ger['OUTRAS PARTES / NÃO-CLIENTES'].str.contains('ZURICH', na=False)

# 2. Aplique o filtro usando .loc 
df_zurich = df_ger.loc[condicao_1].copy()

# 3. Verifique
df_zurich.head()

In [0]:
df_id_zurick.head()

In [0]:
df_zurich = df_zurich.merge(
    df_id_zurick[['ID DO PROCESSO', 'MATRIZ SEGURO']], 
    left_on='PROCESSO - ID',
    right_on='ID DO PROCESSO',
    how='left'
)

df_zurich = df_zurich.drop(columns=['ID DO PROCESSO'])

In [0]:
nao_pagar = 	    ['PARCEIRO - CONSUMIDOR NÃO ACEITA REPARO OU RECUSA INDENIZAÇÃO','PARCEIRO - DEMORA NA REGULAÇÃO DO SINISTRO',
				    'PARCEIRO - INDENIZAÇÃO INDEVIDA OU INSUFICIENTE DO SINISTRO', 'PARCEIRO - LIMITAÇÃO INDEVIDA DE COBERTURAS DO SEGURO (EX. FURTO QUALIFICADO)',
				    'PARCEIRO - RECUSA OU ATENDIMENTO INADEQUADO DO SINISTRO']

condz1 = df_zurich['MATRIZ SEGURO'].isin(nao_pagar)

df_zurich['SEGUIR COM O PAGAMENTO DE HONORARIOS'] = np.where(condz1, 'NÃO', '')

In [0]:
df_zurich.head()

In [0]:
cond_vazio = df_zurich['MATRIZ SEGURO'].isna() | (df_zurich['MATRIZ SEGURO'] == '')

# Verifica se está na lista de exclusão
cond_nao_pagar = df_zurich['MATRIZ SEGURO'].isin(nao_pagar)

# 2. Defina as listas de Condições e Escolhas (na mesma ordem)
condicoes = [
    cond_vazio,      # Prioridade 1: Se for vazio
    cond_nao_pagar   # Prioridade 2: Se estiver na lista 'nao_pagar'
]

escolhas = [
    'NÃO SE APLICA', # Resultado para vazio
    'NÃO'            # Resultado se estiver na lista
]

# 3. Aplica a lógica com np.select
# O argumento 'default' é o que acontece se NENHUMA condição acima for atendida (ou seja, não está na lista)
df_zurich['SEGUIR COM O PAGAMENTO DE HONORARIOS'] = np.select(condicoes, escolhas, default='SIM')

In [0]:
df_zurich.head(10)

In [0]:
df_zurich.shape

In [0]:
df_ger = df_ger.merge(
    df_zurich[['PROCESSO - ID', 'SEGUIR COM O PAGAMENTO DE HONORARIOS']],
    on='PROCESSO - ID', 
    how='left'
)

In [0]:
conds = df_ger['SEGUIR COM O PAGAMENTO DE HONORARIOS'].isnull()

# Leia-se: "No df_ger, onde a condição é True, na coluna 'SEGUIR...', receba 'NÃO SE APLICA'"
df_ger.loc[conds, 'SEGUIR COM O PAGAMENTO DE HONORARIOS'] = 'NÃO SE APLICA'


In [0]:
df_ger.head()

In [0]:
df_fatu_hist_spark = spark.read.table("databox.juridico_comum.br_Consolidado_Honorarios")
df_fatu_hist = df_fatu_hist_spark.toPandas()

df_fatu_hist.head()

In [0]:
df_fatu_hist = df_fatu_hist.rename(columns={'processo_id': 'PROCESSO - ID', 'data_confirmacao': 'DATA CONFIRMAÇÃO', 'chave_id_status':'CHAVE_ID_STATUS' })

In [0]:
# Converter as colunas e manter os IDs
coluna_int_1 = 'PROCESSO - ID'
coluna_int_2 = '(Processo) ID'

# (pd.to_numeric é mais seguro que .astype(int) pois lida com erros)
df_ger[coluna_int_1] = pd.to_numeric(df_ger[coluna_int_1], errors='coerce')
df_ac[coluna_int_1] = pd.to_numeric(df_ac[coluna_int_1], errors='coerce')
df_fatu_hist[coluna_int_1] = pd.to_numeric(df_fatu_hist[coluna_int_1], errors='coerce')
df_fatu_hist_old[coluna_int_2] = pd.to_numeric(df_fatu_hist_old[coluna_int_2], errors='coerce')

In [0]:
df_ac = df_ac.drop_duplicates(subset=['PROCESSO - ID'], keep='last')
df_fatu_hist = df_fatu_hist.drop_duplicates(subset=['PROCESSO - ID'], keep='last')
df_fatu_hist_old = df_fatu_hist_old.drop_duplicates(subset=['(Processo) ID'], keep='last')

In [0]:
#Realiza o proc entre a tabela Df_ger e a tabela df_ac, trazendo a DATA ACEITE
df_ger_1 = pd.merge(
    df_ger,
    df_ac[['PROCESSO - ID', 'DATA ACEITE']],
    on='PROCESSO - ID',
    how='left'
)

In [0]:
df_ger_1.head()

In [0]:
df_ger_1 = df_ger_1.rename(columns={'DATA ACEITE': 'DATA DO ACEITE'})

In [0]:
#Realiza o map e completa as linhas nulas do DATA ACEITE, 
# Com a DATA CONFIRMAÇÃO do hist de faturamento onde existe relação entre os IDs

mapa_de_datas = df_fatu_hist.set_index('PROCESSO - ID')['DATA CONFIRMAÇÃO']

df_ger_1['DATA DO ACEITE'] = df_ger_1['DATA DO ACEITE'].fillna(df_ger_1['PROCESSO - ID'].map(mapa_de_datas))


In [0]:
df_ger_1.head()

In [0]:
# Realiza o map e completa as linhas nulas do DATA ACEITE, seguindo a regra de que a DATA ACEITE é null e o STATUS é PRÉ CADASTRO

cond_vazia = df_ger_1['DATA DO ACEITE'].isnull()
cond_status = df_ger_1['STATUS'] == 'PRÉ CADASTRO'

df_ger_1.loc[cond_vazia & cond_status, 'DATA DO ACEITE'] = "sem data aceite"

In [0]:
#primeiro será necessário converter a coluna DATA REGISTRO para datetime

df_ger_1['DATA REGISTRADO'] = pd.to_datetime(df_ger_1['DATA REGISTRADO'], errors='coerce')
cond_vazia = df_ger_1['DATA DO ACEITE'].isnull()

#Segundo passo é filtrar os registros que estão no mês SELECIONADO
cond_meses = df_ger_1['DATA REGISTRADO'].dt.month.isin([11,12,1]) #<===== MUDAR TODOS OS MESES A MEDIDA DO MES DO FATURAMENTO

#terceiro passo é aplicar a regra de que a DATA ACEITE é null e a data registro está entre os meses SELECIONADOS
df_ger_1.loc[cond_vazia & cond_meses, 'DATA DO ACEITE'] = "sem data aceite"

In [0]:
#Agora os que subraram como null pela regra devem ser clasificados como "com data aceite"
df_ger_1['DATA DO ACEITE'] = df_ger_1['DATA DO ACEITE'].fillna("com data aceite")

In [0]:
#ESCREVE NA NOVA COLUNA STATUS_1 TUDO QUE ESTIVER SEM DATA ACEITE ESCREVE NA COLUNA STATUS_1 = 'Não Faturar - Sem Data Aceite'

df_ger_1['STATUS_1'] = ''
cond1 = df_ger_1['DATA DO ACEITE'] == 'sem data aceite'
df_ger_1.loc[cond1, 'STATUS_1'] = 'Não Faturar - Sem Data Aceite'

In [0]:
id_str = df_ger_1['PROCESSO - ID'].astype(str).str.replace(r'\.0$', '', regex=True)
escrit_str = df_ger_1['ESCRITÓRIO EXTERNO'].astype(str)

df_ger_1['Nova_Coluna'] = (
    id_str + 'ENCERRADO' + escrit_str
)


In [0]:
df_ger_1.head()

In [0]:
#renomeia coluna concatenada para não dar erro

df_fatu_hist.rename(columns={'Chave (ID + STATUS)': 'CHAVE_ID_STATUS'}, inplace=True)

df_fatu_hist.columns = df_fatu_hist.columns.str.strip()

In [0]:
#Dropa as chaves iguais mantandendo a última atualização

df_fatu_hist = df_fatu_hist.drop_duplicates(subset=['CHAVE_ID_STATUS'], keep='last')

In [0]:
#Traz a coluna CHAVE_ID_STATUS e relaciona as correpondencias, se nãp houver, o valor fica nulo
df_resumido = df_fatu_hist[['CHAVE_ID_STATUS']]
df_ger_1 = pd.merge(
    df_ger_1,
    df_resumido,
    left_on='Nova_Coluna',
    right_on='CHAVE_ID_STATUS',
    how='left'
)

In [0]:
df_ger_1.head()

In [0]:
#Dropa as duplicada do vaturamento antigo
df_fatu_hist_old['Chave (ID + STATUS)_1'] = df_fatu_hist_old['Chave (ID + STATUS)']

In [0]:
#Realiza o map e grava nas linhas vazias do CHAVE_ID_STATUS 
df_fatu_hist_old = df_fatu_hist_old.drop_duplicates(subset=['Chave (ID + STATUS)'], keep='last')

mapa_old = df_fatu_hist_old.set_index('Chave (ID + STATUS)')['Chave (ID + STATUS)_1']

df_ger_1['CHAVE_ID_STATUS'] = df_ger_1['CHAVE_ID_STATUS'].fillna(
    df_ger_1['Nova_Coluna'].map(mapa_old)
)

In [0]:
#Aplica a condição de escrita na coluna STATUS_1
cond2 = df_ger_1['CHAVE_ID_STATUS'].notnull() & df_ger_1['Nova_Coluna'].notnull()
df_ger_1.loc[cond2, 'STATUS_1'] = 'Não Faturar - Encerramento pago em outro faturamento'

#dropa as colunas criadas para validação
df_ger_1 = df_ger_1.drop(columns=['Nova_Coluna', 'CHAVE_ID_STATUS'])

In [0]:
df_ger_1.head()

In [0]:
#Condição de não faturar Zurick e Faturar o restante que foi null 
cond3 = df_ger_1['SEGUIR COM O PAGAMENTO DE HONORARIOS'] == 'NÃO'
df_ger_1.loc[cond3, 'STATUS_1'] = 'Não Faturar - Zurick'

# A condição correta para encontrar CÉLULAS VAZIAS (strings vazias)
cond4 = df_ger_1['STATUS_1'] == ''

# O resto do seu código está perfeito e vai funcionar com a condição correta
df_ger_1.loc[cond4, 'STATUS_1'] = 'Faturar'

In [0]:
# Mostra um array com cada valor único que existe na coluna 'STATUS'
valores_unicos_status = df_ger_1['STATUS_1'].unique()

print("Valores únicos na coluna 'STATUS':")
print(valores_unicos_status)

In [0]:
#Reordena as colunas para ficar igual ao original

reordena_colunas = df_ger_1.columns.to_list()
reordena_colunas.remove('STATUS_1')
reordena_colunas.insert(0, 'STATUS_1')
reordena_colunas.remove('DATA DO ACEITE')
reordena_colunas.insert(10,'DATA DO ACEITE')

df_ger_1 = df_ger_1[reordena_colunas]

In [0]:
df_ger_1.columns

In [0]:
colunas_para_manter = [
    'STATUS_1', 'PROCESSO - ID', 'PASTA', 'GRUPO', 'ÁREA DO DIREITO',
    'SUB-ÁREA DO DIREITO', 'ESTEIRA', 'ASSUNTO (CÍVEL) - PRINCIPAL',
    'ASSUNTO (CÍVEL) - ASSUNTO', 'DATA REGISTRADO', 'DATA DO ACEITE', 'STATUS',
    'CENTRO DE CUSTO / ÁREA DEMANDANTE - CÓDIGO', 'EMPRESA',
    'ADVOGADO RESPONSÁVEL', 'VALOR DA CAUSA', 'ESFERA', 'ESTADO', 'COMARCA',
    'FORO/TRIBUNAL/ÓRGÃO', 'VARA/ÓRGÃO', 'AÇÃO', 'DISTRIBUIÇÃO',
    'NÚMERO DO PROCESSO', 'PARTE CONTRÁRIA NOME', 'PARTE CONTRÁRIA CPF',
    'ESCRITÓRIO EXTERNO', 'Escritório Anterior', 'FASE',
    'COMPRA DIRETA OU MARKETPLACE? - NEW ', 'USUÁRIO CADASTRANTE',
    'DATA DA BAIXA PROVISÓRIA', 'DATA DA REATIVAÇÃO', 'VALOR PROVISIONADO',
    'VALOR PROVISIONADO ATUALIZADO', 'PROCESSO CRITICO? (CÍVEL)',
    'ASSISTENTE JURÍDICO - CÍVEL', 'ASSISTENTE JURÍDICO -',
    'DATA AUDIENCIA INICIAL', 'RECLAMAÇÃO BLACK FRIDAY?',
    'OUTRAS PARTES / NÃO-CLIENTES', 'MOTIVO DA CRITICIDADE',
    'ADVOGADO DA PARTE CONTRÁRIA', 'DATA DE REABERTURA',
    'DATA DE RECEBIMENTO', 'PARTE DESCRITA NA INICIAL',
    'TIPO DE CONTINGÊNCIA', 'ANO DA BLACK FRIDAY',
    'DATA DA ÚLTIMA MOVIMENTAÇÃO PROCESSUAL', 'OBSERVAÇÕES', 'OBSERVAÇÃO',
    'TERCEIRO SOLVENTE', 'INCONTROVERSO (ZERAR PROVISÃO)', 'RESULTADO',
    'DATA DE SOLICITAÇÃO DE ENCERRAMENTO',
    'DATA DE ENCERRAMENTO NO TRIBUNAL', 'MOTIVO DE ENCERRAMENTO ',
    'RECLAMAÇÃO ANTERIOR NO PROCON?', 'DATA DE REGISTRO DO ENCERRAMENTO',
    'EXISTE PROCESSO COM O MESMO CPF?', 'CÍVEL MASSA - RESPONSABILIDADE',
    'CÍVEL MASSA - TIPOS DE TERCEIRO',
    'SEGUIR COM O PAGAMENTO DE HONORARIOS'
]

# 2. Crie um novo DataFrame que é uma "fatia" do original, 
df_ger_1 = df_ger_1[colunas_para_manter]

# Agora, 'df_filtrado' contém APENAS as colunas que você listou.
print(df_ger_1.head())

In [0]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter
import shutil 

# --- CAMINHOS E NOMES ---
caminho_modelo = '/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Base_Gerencial_Modelo/CIVEL_GERENCIAL.xlsx'
caminho_final = '/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Gerencial_Consolidada_Final/'
nome_do_arquivo = f'BASE_GERENCIAL - {Mes}_Consolidada.xlsx'
caminho_saida = caminho_final + nome_do_arquivo

nome_da_planilha = 'CÍVEL' 

caminho_temporario = f'/tmp/{nome_do_arquivo}'

# --- LINHA INICIAL ---
linha_cabecalho = 6
linha_inicio_dados = linha_cabecalho + 1

try:
    workbook = load_workbook(caminho_modelo)
    
    sheet = workbook[nome_da_planilha]
    
    rows = dataframe_to_rows(df_ger_1, index=False, header=False)

    for r_idx, row in enumerate(rows, start=linha_inicio_dados):
        for c_idx, value in enumerate(row, start=1):
            sheet.cell(row=r_idx, column=c_idx, value=value)

    if sheet.auto_filter:
        max_col_letra = get_column_letter(sheet.max_column)
        novo_intervalo_filtro = f"A{linha_cabecalho}:{max_col_letra}{sheet.max_row}"
        sheet.auto_filter.ref = novo_intervalo_filtro

    # Etapa 1: Salvar no local temporário
    print(f"Salvando arquivo temporariamente em: {caminho_temporario}")
    workbook.save(caminho_temporario)

    # Etapa 2: Mover o arquivo para o destino final no Volume
    print(f"Movendo arquivo para o destino final: {caminho_saida}")
    shutil.move(caminho_temporario, caminho_saida)

    print("Arquivo salvo com sucesso no Databricks!")
    print(f"Caminho final: {caminho_saida}")

except Exception as e:
    print(f"Ocorreu um erro: {e}")

In [0]:
# CONDIÇÃO INVERSA: Contar quantas linhas SÃO 'Não Faturar - Sem Data Aceite'
#condicao_status_igual = df_ger_1['STATUS_1'] == 'Não Faturar - Sem Data Aceite'
#contagem_status_igual = condicao_status_igual.sum()

#print(f"Total de linhas onde o STATUS_1 É 'Não Faturar - Sem Data Aceite': {contagem_status_igual}")

In [0]:
#nulos_total = df_ger_1['DATA ACEITE'].isnull().sum()
#print(f'Total de linhas com nulos na coluna "DATA ACEITE": {nulos_total}')